<a href="https://colab.research.google.com/github/adibansal06/UCS420_CC_LabAssignments_1024170054/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re

In [3]:
roll_number = "1024170054"

# Q1: Build Personalized Knowledge Base

In [4]:
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

categories = ["billing", "account", "general"]

last_two_digits = [int(d) for d in roll_number[-2:]]

personalized_entries = []

# Digit 5 -> 5 % 3 = 2 -> general
personalized_entries.append({
    "question": "how can i check the available services",
    "answer": "You can check all available services from the Services section.",
    "keywords": "services available options",
    "category": categories[last_two_digits[0] % 3]
})

# Digit 4 -> 4 % 3 = 1 -> account
personalized_entries.append({
    "question": "how do i update my registered mobile number",
    "answer": "Go to Account Settings and update your registered mobile number.",
    "keywords": "mobile number update",
    "category": categories[last_two_digits[1] % 3]
})

faq_data = fixed_entries + personalized_entries

df = pd.DataFrame(faq_data)

print("Q1: Final 6-row FAQ DataFrame")
print(df)
print()

Q1: Final 6-row FAQ DataFrame
                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4       how can i check the available services   
5  how do i update my registered mobile number   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can check all available services from the ...   
5  Go to Account Settings and update your registe...   

                     keywords category  
0       fee cost price charge  billing  
1        password reset login  account  
2      hours timing open time  general  
3         pay payment upi fee  billing  
4  services available

# Q2: Generate and Score Hypotheses

In [6]:
def normalize(word):
    """Remove punctuation and convert a word to lowercase."""
    return re.sub(r"[^a-z0-9]", "", word.lower())


def score_hypotheses(query, df):
    query_words = set(
        normalize(word)
        for word in query.split()
        if normalize(word)
    )

    results = []

    for _, row in df.iterrows():
        faq_words = set(
            normalize(word)
            for word in row["keywords"].split()
            if normalize(word)
        )

        overlap = query_words & faq_words
        score = len(overlap)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    # Highest confidence first
    results.sort(key=lambda x: x["score"], reverse=True)

    return results


query = input("Q2: Enter your query: ")

results = score_hypotheses(query, df)

print("\nMatching hypotheses ranked by confidence:")

if results:
    for result in results:
        print(
            f"Score: {result['score']} | "
            f"Category: {result['category']} | "
            f"Question: {result['question']}"
        )
        print(f"Answer: {result['answer']}")
else:
    print("No matching FAQ entry found.")

print()

Q2: Enter your query: what are your working hours

Matching hypotheses ranked by confidence:
Score: 1 | Category: general | Question: what are your working hours
Answer: We are open 9 AM to 5 PM.



# Q3: Find FAQs from the same category

In [7]:
def same_category(category_name, df):
    return df[df["category"] == category_name][["question", "category"]]


personalized_category = personalized_entries[0]["category"]

print("Q3: FAQs in the same category")
print(f"Category selected: {personalized_category}")
print(same_category(personalized_category, df))
print()

Q3: FAQs in the same category
Category selected: general
                                 question category
2             what are your working hours  general
4  how can i check the available services  general



# Q4: Add a new keyword and save CSV

In [8]:
print("Q4: Add a new keyword")

entry_index = 0

print("Selected FAQ:")
print(df.loc[entry_index, "question"])

new_keyword = input("Enter a new keyword: ").strip()

if new_keyword:
    current_keywords = df.loc[entry_index, "keywords"]

    if new_keyword not in current_keywords.split():
        df.loc[entry_index, "keywords"] = (
            current_keywords + " " + new_keyword
        )
        print("Keyword added successfully.")
    else:
        print("Keyword already exists.")
else:
    print("No keyword entered.")

csv_file = f"{roll_number}_faq_data.csv"
df.to_csv(csv_file, index=False)

print(f"Updated DataFrame saved as {csv_file}")
print()

Q4: Add a new keyword
Selected FAQ:
what is the annual fee
Enter a new keyword: Hello
Keyword added successfully.
Updated DataFrame saved as 1024170054_faq_data.csv



# Q5: Count FAQ entries per category

In [9]:
print("Q5: Number of FAQ entries per category")

category_counts = df.groupby("category").size()

print(category_counts)
print()

Q5: Number of FAQ entries per category
category
account    2
billing    2
general    2
dtype: int64



# Q6: Handle ties for highest score

In [10]:
def find_best_matches(query, df):
    results = score_hypotheses(query, df)

    if not results:
        print("No matching FAQ entry found.")
        return

    highest_score = results[0]["score"]

    best_matches = [
        result for result in results
        if result["score"] == highest_score
    ]

    print(f"Highest confidence score: {highest_score}")

    if len(best_matches) > 1:
        print("Tie detected! All equally matching entries are:")
    else:
        print("Best matching entry:")

    for result in best_matches:
        print(f"\nQuestion: {result['question']}")
        print(f"Answer: {result['answer']}")
        print(f"Category: {result['category']}")
        print(f"Score: {result['score']}")


# Demonstration 1: Tie
print("Q6: Tie demonstration")
print("Query: fee")
find_best_matches("fee", df)

print("\nQ6: Non-tie demonstration")
print("Query: reset password")
find_best_matches("reset password", df)

Q6: Tie demonstration
Query: fee
Highest confidence score: 1
Tie detected! All equally matching entries are:

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Score: 1

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Score: 1

Q6: Non-tie demonstration
Query: reset password
Highest confidence score: 2
Best matching entry:

Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Score: 2
